# Classification Visit Mode

**Objective:** Predict VisitMode and evaluate it with macro F1 and a confusion matrix.

The code is split into visible, explainable steps for a demonstration video.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'Tourism Dataset'
PROCESSED = ROOT / 'data' / 'processed'
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
master = pd.read_csv(PROCESSED/'master_dataset.csv')
FEATURES = ['VisitYear','VisitMonth','UserContinent','UserRegion','UserCountry','UserCity','AttractionType','AttractionCity','AttractionCountry','AttractionRegion','AttractionContinent']
X, y = master[FEATURES].fillna('Unknown'), master['VisitMode'].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=RANDOM_STATE, stratify=y)
y.value_counts()

In [ ]:
categorical = [c for c in FEATURES if c not in ['VisitYear','VisitMonth']]
preprocess = ColumnTransformer([('categorical', OneHotEncoder(handle_unknown='ignore'), categorical), ('numeric', 'passthrough', ['VisitYear','VisitMonth'])])
classifier = Pipeline([('preprocessing', preprocess), ('model', RandomForestClassifier(n_estimators=60, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
classifier.fit(X_train, y_train)
pred = classifier.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test,pred), 3))
pd.DataFrame(classification_report(y_test,pred,output_dict=True,zero_division=0)).T

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Visit Mode Confusion Matrix'); plt.show()
joblib.dump(classifier, ROOT/'models'/'classification'/'visit_mode_model.pkl')